# Building the deck tables

Turns the decklist text files in `data/external/decks/` into three tables in `data/processed/`. It needs `cards.parquet` and `moves.parquet` to exist already.

| Table | One row per | Main columns |
|---|---|---|
| `decks` | deck | `deck_id`, `mode`, `n_cards`, `n_distinct`, `n_lines`, `unresolved_weight`, `source` |
| `deck_cards` | card in a deck | `deck_id`, `card_id`, `copies`, `avg_count` |
| `deck_profile` | card seen in a deck's list | `deck_id`, `card_id`, `avg_count` |

`deck_id` is the filename without `.txt`. To add a deck, save its list as a new `.txt` file in the folder and re-run the notebook.

In [6]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("/Users/tam_pr/tcg_bot")
PROC = ROOT / "data" / "processed"
DECK_DIR = ROOT / "data" / "external" / "decks"

cards = pd.read_parquet(PROC / "cards.parquet")
moves = pd.read_parquet(PROC / "moves.parquet")

## 1. Parsing a deck file

A deck file has one card per line: a count, the card name, and optionally the set code and card number.

```
# source: limitlesstcg.com, 2026-11-23     <- '#' starts a comment
Pokémon: 16
3 Riolu MEG 76
4 Ultra Ball
```

- Section headers (`Pokémon: 16` or `Pokémon (16.47)`) only mark the section. The numbers in them are ignored.
- Blank lines and a `Total Cards:` footer are skipped.
- Set and number are optional. They are only needed for cards that have several versions (for example Riolu or Shaymin).
- A line that can't be read raises an error with its line number.

In [7]:
def parse_limitless(text):
    section, rows = None, []
    for n, line in enumerate(text.splitlines(), 1):
        line = line.split("#")[0].strip()                       # '#' starts a comment
        if not line or line.lower().startswith("total cards"):  # blank lines and the "Total Cards: 60" footer
            continue
        m = re.match(r"^(Pokémon|Trainer|Energy)\s*[:(]\s*[\d.]+\)?$", line)   # section header
        if m:
            section = m.group(1)
            continue
        m = re.match(r"^(\d+(?:\.\d+)?)\s+(.+?)(?:\s+([A-Z0-9][A-Z0-9-]{1,5})\s+(\d+))?$", line)
        if not m:
            raise ValueError(f"line {n}: can't read {line!r}")
        rows.append({"section": section, "avg_count": float(m[1]), "name": m[2],
                     "set": m[3], "number": int(m[4]) if m[4] else None})
    return pd.DataFrame(rows)

## 2. Resolving lines to cards

Every line has to become a `card_id` from the `cards` table. `resolve` tries, in order:

1. **`print_alias`**: an exact `(name, set, number)` entry in `PRINT_ALIASES`.
2. **`name`**: the dataset has exactly one card with that name (after applying `ALIASES`).
3. **`set+number`**: the name has several versions, so the set and number pick one.

Anything else is `ambiguous` (several versions and no match) or `unmatched` (no such card).

- `ALIASES` maps names Limitless writes differently from our dataset, for example `Fighting Energy` to `Basic {F} Energy`.
- `PRINT_ALIASES` maps a specific printing our dataset doesn't list to a card. **Each entry is a hand-made assumption. Check it against the real card.** A wrong entry silently puts the wrong card in the deck.

In [8]:
# Limitless name -> name used in our dataset
ALIASES = {
    "Fighting Energy": "Basic {F} Energy",
    "Rocky Fighting Energy": "Rock Fighting Energy",
}
# (lowercase name, set, number) -> card_id, for prints our dataset doesn't list. CHECK EACH ONE BY HAND.
PRINT_ALIASES = {
    ("snorunt", "ASC", 227): 860,        # assumed alternate print of Snorunt ASC 46 (70 HP, Chilly)
    ("chien-pao", "PR-SV", 152): 209,    # assumed Chien-Pao SSP 56 (Water, Snow Sink); POR 54 is Darkness
}

def resolve(raw, cards):
    ids_by_name = cards.groupby(cards["card_name"].str.lower())["card_id"].agg(list)
    out_id, how = [], []
    for r in raw.itertuples():
        key = (r.name.lower(), r.set, r.number)
        if key in PRINT_ALIASES:
            out_id.append(PRINT_ALIASES[key]); how.append("print_alias")
            continue
        ids = ids_by_name.get(ALIASES.get(r.name, r.name).lower(), [])
        if len(ids) == 1:                       # one card with that name
            out_id.append(ids[0]); how.append("name")
        elif len(ids) > 1:                      # several versions: use set + number
            hit = cards[cards["card_id"].isin(ids)
                        & (cards["expansion"] == r.set) & (cards["collection_no"] == r.number)]
            out_id.append(hit["card_id"].iloc[0] if len(hit) == 1 else pd.NA)
            how.append("set+number" if len(hit) == 1 else "ambiguous")
        else:
            out_id.append(pd.NA); how.append("unmatched")
    return raw.assign(card_id=pd.array(out_id, dtype="Int64"), matched_by=how)

## 3. From lines to a 60-card deck

`build_deck` first adds up lines that point to the same card (different prints), then works in one of two modes:

- **exact**: all counts are whole numbers. The list is used exactly as written. Nothing is added or dropped, and an unresolved line or a total other than 60 stops the build with an error.
- **average**: counts have decimals, as on a Limitless average list. The counts are rounded down, then the leftover copies go to the largest fractions. At most 4 copies of a card (Basic Energy is exempt) and at most one ACE SPEC are allowed. Up to 0.5 cards of unresolved weight are tolerated.

The mode is stored in the `decks` table.

In [9]:
def build_deck(resolved, cards, total=60, max_copies=4, tolerance=0.5):
    """Whole-number list: used exactly as written (never padded or guessed).
    Average list (fractional counts): rounded to a legal 60."""
    missing = resolved.loc[resolved["card_id"].isna()]
    df = resolved[resolved["card_id"].notna()].copy()
    df["card_id"] = df["card_id"].astype(int)
    df = df.groupby("card_id", as_index=False)["avg_count"].sum()   # same card on several lines (prints)
    df = df.merge(cards[["card_id", "card_name", "family", "card_kind", "is_ace_spec"]], on="card_id")
    is_average = bool((resolved["avg_count"] % 1 != 0).any())

    if not is_average:
        if len(missing):
            raise ValueError("unresolved lines in a whole-number list:\n"
                             + missing[["avg_count", "name", "set", "number", "matched_by"]].to_string(index=False))
        df["copies"] = df["avg_count"].astype(int)
        if df["copies"].sum() != total:
            raise ValueError(f"deck has {df['copies'].sum()} cards, expected {total}")
        return df, "exact"

    if missing["avg_count"].sum() > tolerance:
        raise ValueError("too much of the average list is unresolved:\n"
                         + missing[["avg_count", "name", "set", "number", "matched_by"]].to_string(index=False))
    cap = np.where(df["card_kind"] == "Basic Energy", 10**6, max_copies)
    df["copies"] = np.minimum(np.floor(df["avg_count"]).astype(int), cap)
    need = total - df["copies"].sum()
    ace_in = int(df.loc[df["is_ace_spec"], "copies"].sum())
    frac = df["avg_count"] - np.floor(df["avg_count"])
    for i in frac.sort_values(ascending=False).index:        # extra copies go to the biggest fractions
        if need == 0 or frac[i] <= 0:
            break
        if df.at[i, "copies"] >= cap[df.index.get_loc(i)]:
            continue
        if df.at[i, "is_ace_spec"] and ace_in >= 1:
            continue
        df.at[i, "copies"] += 1
        need -= 1
        ace_in += int(df.at[i, "is_ace_spec"])
    if need != 0:
        raise ValueError(f"could not reach {total} cards, still need {need}")
    return df[df["copies"] > 0].reset_index(drop=True), "average"

## 4. Deck rules

`validate_deck` checks the final list:

- exactly 60 cards;
- at most 4 copies of a card by name (Basic Energy is exempt);
- at most 1 ACE SPEC card;
- at least 1 Basic Pokémon.

These are the rules from the pipeline plan. They still need to be checked against the official rulebook.

In [10]:
def validate_deck(deck, total=60, max_copies=4):
    by_name = deck.groupby("card_name").agg(
        copies=("copies", "sum"),
        basic_energy=("card_kind", lambda s: (s == "Basic Energy").all()))
    assert deck["copies"].sum() == total, deck["copies"].sum()
    over = by_name[(~by_name["basic_energy"]) & (by_name["copies"] > max_copies)]
    assert over.empty, f"more than {max_copies} copies:\n{over}"
    ace = deck[deck["is_ace_spec"]]
    assert ace["copies"].sum() <= 1, f"more than one ACE SPEC:\n{ace[['card_name', 'copies']]}"
    assert (deck["card_kind"] == "Basic Pokémon").any(), "no Basic Pokémon in the deck"

## 5. Building every deck file

Loops over the `.txt` files in the folder, then parses, resolves, builds and validates each deck.

- It prints one row per deck with how its lines were matched (`name`, `set+number`, `print_alias`, `unmatched`...).
- If a deck fails, the reason is printed under `FAILED`, and a `RuntimeError` stops the notebook before anything is saved.
- `unresolved_weight` in `decks` is the number of cards that couldn't be matched. It should be 0 for an exact list.

In [11]:
decks, deck_cards, deck_profile, errors, reports = [], [], [], {}, []
for path in sorted(DECK_DIR.glob("*.txt")):
    deck_id = path.stem
    try:
        raw = parse_limitless(path.read_text(encoding="utf-8"))
        resolved = resolve(raw, cards)
        deck, mode = build_deck(resolved, cards)
        validate_deck(deck)
    except (ValueError, AssertionError) as e:
        errors[deck_id] = str(e)
        continue
    bad = resolved[resolved["card_id"].isna()]
    reports.append({"deck_id": deck_id, "lines": len(raw), **resolved["matched_by"].value_counts().to_dict()})
    profile = (resolved[resolved["card_id"].notna()].assign(card_id=lambda d: d["card_id"].astype(int))
               .groupby("card_id", as_index=False)["avg_count"].sum().assign(deck_id=deck_id))
    deck_profile.append(profile[["deck_id", "card_id", "avg_count"]])
    deck_cards.append(deck.assign(deck_id=deck_id)[["deck_id", "card_id", "copies", "avg_count"]])
    decks.append({"deck_id": deck_id, "mode": mode, "n_cards": int(deck["copies"].sum()),
                  "n_distinct": len(deck), "n_lines": len(raw),
                  "unresolved_weight": round(float(bad["avg_count"].sum()), 2),
                  "source": "limitless_average" if mode == "average" else "decklist"})

print(pd.DataFrame(reports).fillna(0).astype({c: int for c in pd.DataFrame(reports).columns if c != "deck_id"}, errors="ignore"))
for deck_id, msg in errors.items():
    print(f"\nFAILED  {deck_id}\n{msg}")
if errors:
    raise RuntimeError(f"fix {list(errors)} before saving")

decks = pd.DataFrame(decks)
deck_cards = pd.concat(deck_cards, ignore_index=True)
deck_profile = pd.concat(deck_profile, ignore_index=True)
decks

        deck_id  lines  name  set+number  print_alias
0       lopunny     31    25           4            2
1  mega_lucario     22    21           1            0


,deck_id,mode,n_cards,n_distinct,n_lines,unresolved_weight,source
0,lopunny,exact,60,26,31,0.0,decklist
1,mega_lucario,exact,60,22,22,0.0,decklist


## 6. Reading the decks back

Joins `deck_cards` to `cards` to print each deck by card name, sorted by card kind. Use this to check the lists against the original files.

In [12]:
view = deck_cards.merge(cards[["card_id", "card_name", "card_kind", "expansion", "collection_no"]], on="card_id")
for deck_id, g in view.groupby("deck_id"):
    print(f"\n== {deck_id}  ({g['copies'].sum()} cards)")
    print(g.sort_values(["card_kind", "copies"], ascending=[True, False])
           [["copies", "card_name", "card_kind", "expansion", "collection_no"]].to_string(index=False))


== lopunny  (60 cards)
 copies              card_name       card_kind expansion  collection_no
      3       Basic {W} Energy    Basic Energy       SVE              3
      2              Dunsparce   Basic Pokémon       JTG            120
      2                Buneary   Basic Pokémon       PFL             83
      2                Snorunt   Basic Pokémon       ASC             46
      1              Dunsparce   Basic Pokémon       TEF            128
      1              Fan Rotom   Basic Pokémon       SCR            118
      1              Chien-Pao   Basic Pokémon       SSP             56
      1                Psyduck   Basic Pokémon       ASC             39
      4     Buddy-Buddy Poffin            Item       TEF            144
      4               Poké Pad            Item       POR             81
      3             Ultra Ball            Item       SVI            196
      2        Night Stretcher            Item       SFA             61
      2           Pokégear 3.0          

## 7. Scope and saving

- `in_scope` counts the rows of `moves` whose card appears in at least one deck. This is the amount of effect text the engine has to support first.
- The three tables are saved to `data/processed/`: `decks.parquet`, `deck_cards.parquet` and `deck_profile.parquet`.

`deck_cards` is the playable list. `deck_profile` keeps the unrounded `avg_count` for each card, and the opponent-inference step will use it later.

In [13]:
in_scope = moves["card_id"].isin(deck_cards["card_id"])
print(in_scope.sum(), "of", len(moves), "moves are in at least one playable deck")

decks.to_parquet(PROC / "decks.parquet", index=False)
deck_cards.to_parquet(PROC / "deck_cards.parquet", index=False)
deck_profile.to_parquet(PROC / "deck_profile.parquet", index=False)

54 of 2265 moves are in at least one playable deck
